# CPU Baseline — CFPB Consumer Complaints Pipeline

**Course:** Engineering of Data Analysis (2779), T4 2025-26, NOVA SBE  
**Deadline:** 27 April 2026

---

> **Disclaimer:** This notebook is a condensed version of the original ML assignment notebook (`72782_Complaints_Notebook.ipynb`). It retains only the compute-relevant stages of the pipeline — feature engineering, model training, and inference — specifically instrumented for wall-clock timing. The full exploratory data analysis (Q1), model evaluation curves (Q3), deployment preparation (Q4), and strategic insights (Q5) are contained in the original notebook and are not reproduced here.
>
> Timing instrumentation has been added around feature engineering, Logistic Regression cross-validation and inference, XGBoost cross-validation and inference, and a dataset-size sweep (Section 8). All timings are recorded in seconds using `time.perf_counter()`. These results feed directly into `GPU_Solution.ipynb` for speedup comparison.
>
> No modelling logic, hyperparameter grids, or random seeds have been altered relative to the original submission.


# 1. Setup & Data Loading

This section imports all required libraries and loads the raw CFPB complaint dataset. The library set is identical to the original notebook. `numpy.random.seed(42)` is set globally to ensure deterministic behaviour across all subsequent operations.


In [ ]:
# Importing necessary libraries

import numpy as np
import pandas as pd
import time

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score)

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)


In [ ]:
# fixing random seed globally to ensure reproducibility

np.random.seed(42)

## 1.1 Loading the Dataset

The raw dataset (`complaints_training.csv`) contains 321,430 consumer complaint records filed with the CFPB. It is loaded directly from the working directory. The feature matrix and target variable are separated before the train/test split.


In [ ]:
# importing the training data

df_raw = pd.read_csv("complaints_training.csv")

## 2. Feature Engineering

All feature engineering is implemented in a single function `feature_engineering()`. The function accepts a raw dataframe with the original CFPB column names and returns a fully transformed feature matrix ready for modeling. It operates identically on both training data and any unseen data with the same schema. All encoding maps fitted on training data are passed in explicitly during inference, ensuring no leakage.

The final feature set spans five categories:

1. **Temporal signals**: Derived from complaint routing timing
2. **Response quality**: Flags decomposing the company's resolution behavior
3. **Narrative-based behavioral signals**: Extracted from the consumer's free-text submission 
4. **Complaint metadata**: Capturing consumer demographics and case specificity 
5. **Target- and frequency-encoded representations** of high-cardinality categoricals

Preprocessing is handled within the same function and the model pipelines. Missing values in categorical features are imputed with the string `'Missing'` prior to encoding, preserving missingness as a distinct category that allows the model to learn a separate dispute rate for records where that feature was not reported, rather than discarding incomplete records or collapsing them into the global mean. Numeric features derived from dates or text lengths are zero-imputed where a missing value represents genuine absence (e.g., no narrative submitted). Any residual null values that survive feature engineering are caught by a `SimpleImputer` (median strategy) as the first step in both model pipelines. For Logistic Regression, features are standardized via `StandardScaler` to ensure regularization penalizes coefficients on a comparable scale; XGBoost is scale-invariant and requires no standardization.

Target encoding uses a smoothed estimator with a smoothing factor of 20, shrinking rare category estimates toward the global mean to reduce overfitting. All encodings are fitted exclusively on training data and applied to the test set via stored mappings, mirroring the deployment scenario where unseen companies, issues, or ZIP regions must fall back to the global mean gracefully.

**Final feature set**

| Feature | Category | Rationale |
|---|---|---|
| `response_lag_days` | Temporal | Days between complaint receipt and company forwarding, clipped at 120. Longer delays signal routing bottlenecks that increase consumer frustration. |
| `received_year` | Temporal | Captures year-level temporal trends in dispute rates as CFPB processes evolved across 2014-2015. |
| `day_of_week` | Temporal | Submission timing proxy; complaints filed on different days may reflect different consumer intent or urgency. |
| `company_response_monetary` | Response quality | Binary flag for monetary relief offered. Strong negative predictor of escalation, consumers who receive financial compensation are less likely to dispute. |
| `company_response_non_monetary` | Response quality | Binary flag for non-monetary relief. Captures partial resolution that may still leave consumers dissatisfied. |
| `company_response_explanation` | Response quality | Binary flag for explanation-only responses. Associated with higher dispute rates in Q1 analysis. |
| `company_response_without_relief` | Response quality | Binary flag for cases closed without any form of relief. Distinct resolution state from explanation-only. |
| `company_response_in_progress` | Response quality | Binary flag for unresolved cases. Captures complaints still being processed at prediction time. |
| `company_response_is_relief` | Response quality | Union of monetary and non-monetary relief flags. Single composite signal for whether any substantive relief was provided. |
| `closed_x_timely` | Response quality | Interaction between relief received and timely response. A late resolution may frustrate consumers even when substantive. |
| `timely_response` | Response quality | Binary flag for whether the company responded within the CFPB deadline. |
| `has_public_response` | Response quality | Binary flag for whether the company issued a public-facing statement. Associated with lower dispute rates in Q1 analysis. |
| `company_response_te` | Response quality | Target-encoded dispute rate per response category. Smoother representation of the response type signal complementing the binary flags. |
| `response_te_x_relief` | Response quality | Interaction between target-encoded response rate and relief flag. Sharpens the boundary between high-risk companies that provided relief versus those that did not. |
| `has_narrative` | Narrative | Binary flag for whether the consumer submitted a free-text description. Engaged consumers who write narratives are more likely to follow through with a dispute. |
| `word_count` | Narrative | Length of narrative in words. Longer complaints may indicate more complex or contentious cases. |
| `char_count` | Narrative | Length of narrative in characters. Captures punctuation-heavy complaints that word count may underweight. |
| `avg_word_length` | Narrative | Proxy for linguistic sophistication and formality of the complaint. |
| `narrative_short` | Narrative | Binary flag for narratives under 20 words. Very short narratives may reflect perfunctory submissions with lower escalation intent. |
| `narrative_long` | Narrative | Binary flag for narratives over 200 words. Lengthy complaints may indicate highly engaged consumers building a case for escalation. |
| `escalation_term_count` | Narrative | Count of legal and regulatory keywords such as attorney, lawsuit, fraud, and violation. Direct signal of consumer intent to escalate. |
| `has_escalation_terms` | Narrative | Binary flag derived from `escalation_term_count`. Captures presence of any escalation language regardless of frequency. |
| `narrative_x_no_relief` | Narrative | Interaction between narrative presence and no relief received. Captures the highest-frustration consumer profile. |
| `consent_provided` | Complaint metadata | Binary flag for consumer consent to publish narrative. Consumers who opted in are more engaged and more likely to dispute. |
| `consent_withdrawn` | Complaint metadata | Binary flag for consumers who initially consented then withdrew. Distinct behavioral signal from never consenting. |
| `has_subproduct` | Complaint metadata | Binary flag for sub-product presence. Complaints with more detailed categorization may reflect more complex cases. |
| `has_subissue` | Complaint metadata | Binary flag for sub-issue presence. Same rationale as `has_subproduct`. |
| `tags_any` | Complaint metadata | Binary flag for any consumer tag present. Composite signal for vulnerable consumer populations. |
| `is_older_american` | Complaint metadata | Binary flag for complaints tagged as Older American. Vulnerable population with higher observed dispute rates in Q1. |
| `is_servicemember` | Complaint metadata | Binary flag for complaints tagged as Servicemember. Regulated demographic with distinct complaint handling requirements. |
| `company_volume` | Complaint metadata | Frequency encoding of company. Proxy for company size, larger companies may have more standardized and less consumer-responsive processes. |
| `Product_te` | Encoded categorical | Smoothed dispute rate per product category. Captures product-level escalation risk identified in Q1. |
| `Issue_te` | Encoded categorical | Smoothed dispute rate per issue type. Captures issue-level escalation patterns. |
| `State_te` | Encoded categorical | Smoothed dispute rate per state. Captures regional regulatory environment differences identified in Q1. |
| `Company_te` | Encoded categorical | Smoothed dispute rate per company. Captures company-level complaint handling quality. |
| `Sub_product_te` | Encoded categorical | Smoothed dispute rate per sub-product. More granular product-level signal where available. |
| `Sub_issue_te` | Encoded categorical | Smoothed dispute rate per sub-issue. More granular issue-level signal where available. |
| `Submitted_via_te` | Encoded categorical | Smoothed dispute rate per submission channel. Captures digital vs traditional channel effect identified in Q1. |
| `zip3_region_te` | Encoded categorical | Smoothed dispute rate per three-digit ZIP prefix. More granular regional signal than state-level encoding. |
| `product_x_channel_te` | Encoded categorical | Target-encoded product × submission channel interaction. Captures compound risk patterns not visible in either feature alone. |
| `product_issue_te` | Encoded categorical | Target-encoded product × issue interaction. Captures whether certain issue types are particularly contentious for specific products. |

In [ ]:
def feature_engineering(X, y=None, fit_encoders=None):
    """
    Engineers all features from raw CFPB complaint data.
    
    Parameters
    ----------
    X : pd.DataFrame
        Raw dataframe with original CFPB column names.
    y : pd.Series, optional
        Target variable. Required when fit_encoders is None (training mode).
    fit_encoders : dict, optional
        Pre-fitted encoding maps from training. If None, encoders are fitted
        from scratch (training mode). If provided, encoders are applied as-is
        (inference mode).
    
    Returns
    -------
    out : pd.DataFrame
        Transformed feature matrix ready for modeling.
    encoders : dict
        Fitted encoding maps to be passed during inference.
    """
    encoders = {} if fit_encoders is None else fit_encoders
    training = fit_encoders is None

    # work on a copy to avoid modifying the original
    raw = X.copy()

    # drop columns not needed for feature engineering
    for col in ['Complaint ID', 'Consumer disputed?']:
        if col in raw.columns:
            raw = raw.drop(columns=[col])

    out = pd.DataFrame(index=raw.index)

    # temporal features
    date_rec  = pd.to_datetime(raw['Date received'],        errors='coerce')
    date_sent = pd.to_datetime(raw['Date sent to company'], errors='coerce')

    out['response_lag_days'] = (date_sent - date_rec).dt.days.clip(0, 120).fillna(0).astype(int)
    out['received_year']     = date_rec.dt.year.fillna(2015).astype(int)
    out['day_of_week']       = date_rec.dt.dayofweek.fillna(0).astype(int)

    # response quality features
    resp = raw['Company response to consumer'].fillna('Missing')

    out['company_response_monetary']       = resp.str.contains('monetary',      case=False).astype(int)
    out['company_response_non_monetary']   = resp.str.contains('non-monetary',  case=False).astype(int)
    out['company_response_explanation']    = resp.str.contains('explanation',   case=False).astype(int)
    out['company_response_without_relief'] = resp.str.contains('without relief',case=False).astype(int)
    out['company_response_in_progress']    = resp.str.contains('in progress',   case=False).astype(int)
    out['company_response_is_relief']      = (
        (out['company_response_monetary'] == 1) |
        (out['company_response_non_monetary'] == 1)
    ).astype(int)
    out['timely_response']     = raw['Timely response?'].fillna('No').eq('Yes').astype(int)
    out['closed_x_timely']     = out['company_response_is_relief'] * out['timely_response']
    out['has_public_response'] = raw['Company public response'].notna().astype(int)

    # narrative features
    narr = raw['Consumer complaint narrative'].fillna('')

    out['has_narrative']       = (narr != '').astype(int)
    out['word_count']          = narr.apply(lambda x: len(x.split()) if x else 0)
    out['char_count']          = narr.str.len().fillna(0).astype(int)
    out['avg_word_length']     = narr.apply(
        lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0
    )
    out['narrative_short']     = (out['word_count'].between(1, 19)).astype(int)
    out['narrative_long']      = (out['word_count'] > 200).astype(int)

    escalation_terms = [
        'attorney', 'lawyer', 'lawsuit', 'legal action',
        'sue', 'court', 'cfpb', 'regulation', 'violation',
        'fraud', 'illegal', 'discriminat'
    ]
    esc_pattern = '|'.join(escalation_terms)
    out['escalation_term_count'] = narr.str.lower().str.count(esc_pattern).fillna(0).astype(int)
    out['has_escalation_terms']  = (out['escalation_term_count'] > 0).astype(int)
    out['narrative_x_no_relief'] = out['has_narrative'] * (1 - out['company_response_is_relief'])

    # complaint metadata
    consent = raw['Consumer consent provided?'].fillna('Missing')
    out['consent_provided']  = consent.eq('Consent provided').astype(int)
    out['consent_withdrawn'] = consent.eq('Consent withdrawn').astype(int)
    out['has_subproduct']    = raw['Sub-product'].notna().astype(int)
    out['has_subissue']      = raw['Sub-issue'].notna().astype(int)
    tags = raw['Tags'].fillna('')
    out['tags_any']          = (tags != '').astype(int)
    out['is_older_american'] = tags.str.contains('Older American', case=False).astype(int)
    out['is_servicemember']  = tags.str.contains('Servicemember',  case=False).astype(int)

    # encoding helpers
    def target_encode(series, name, smoothing=20):
        if training:
            tmp = pd.DataFrame({'val': series.values, 'target': y.values},
                               index=series.index)
            agg = tmp.groupby('val')['target'].agg(['mean', 'count'])
            global_mean = y.mean()
            smooth = (agg['count'] * agg['mean'] + smoothing * global_mean) / (
                agg['count'] + smoothing)
            encoders[name] = {'map': smooth.to_dict(), 'global_mean': global_mean}
        return series.map(encoders[name]['map']).fillna(encoders[name]['global_mean'])

    def freq_encode(series, name):
        if training:
            encoders[name] = series.value_counts().to_dict()
        return series.map(encoders[name]).fillna(1)

    # encoded categorical features
    out['Product_te']          = target_encode(raw['Product'].fillna('Missing'),                      'Product_te')
    out['Issue_te']            = target_encode(raw['Issue'].fillna('Missing'),                        'Issue_te')
    out['State_te']            = target_encode(raw['State'].fillna('Missing'),                        'State_te')
    out['Company_te']          = target_encode(raw['Company'].fillna('Missing'),                      'Company_te')
    out['Sub_product_te']      = target_encode(raw['Sub-product'].fillna('Missing'),                  'Sub_product_te')
    out['Sub_issue_te']        = target_encode(raw['Sub-issue'].fillna('Missing'),                    'Sub_issue_te')
    out['Submitted_via_te']    = target_encode(raw['Submitted via'].fillna('Missing'),                'Submitted_via_te')
    out['company_response_te'] = target_encode(raw['Company response to consumer'].fillna('Missing'), 'company_response_te')
    out['zip3_region_te']      = target_encode(
        raw['ZIP code'].fillna('000').astype(str).str[:3],                                            'zip3_region_te')

    prod_x_channel = raw['Product'].fillna('Missing') + '_' + raw['Submitted via'].fillna('Missing')
    out['product_x_channel_te'] = target_encode(prod_x_channel, 'product_x_channel_te')

    prod_x_issue = raw['Product'].fillna('Missing') + '_' + raw['Issue'].fillna('Missing')
    out['product_issue_te'] = target_encode(prod_x_issue, 'product_issue_te')

    out['company_volume'] = freq_encode(raw['Company'].fillna('Missing'), 'company_volume')

    # interaction features
    out['response_te_x_relief'] = out['company_response_te'] * out['company_response_is_relief']

    return out, encoders

## 3. Train/Test Split

I split the dataset into a training set (80%) and a held-out test set (20%) using stratified sampling to preserve the 4:1 class ratio in both partitions. The `feature_engineering()` function is fitted on the training set to derive all encoding maps and then applied to the test set using those stored maps, ensuring that no information from the test set influences the encoding step.

**Note re. time variable in the data**

Since the data spans January 2014 to December 2015, a random split means the model may train on later complaints and be tested on earlier ones, which in a time series context could introduce temporal leakage. I considered a chronological split but opted for stratified random sampling for two reasons. First, the EDA in section 2.5 showed no structural trend or seasonal drift in either complaint volume or dispute rates over the observation period, meaning the data is effectively stationary. Second, the model's temporal features are limited to `day_of_week`, `received_year`, and `response_lag_days`, none of which encode absolute time position, so the model cannot exploit future knowledge in a meaningful way. Under these conditions, stratified random sampling is the more robust choice as it ensures balanced class representation in both partitions without introducing compositional bias from an arbitrary temporal cutoff.

In [ ]:
import time

X = df_raw.drop(columns=['Complaint ID', 'Consumer disputed?'])
y = df_raw['Consumer disputed?'].map({'Yes': 1, 'No': 0})

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

t0 = time.perf_counter()
X_train_enc, encoders = feature_engineering(X_train_raw, y=y_train)
t_fe_train_cpu = time.perf_counter() - t0

t0 = time.perf_counter()
X_test_enc, _         = feature_engineering(X_test_raw, fit_encoders=encoders)
t_fe_test_cpu = time.perf_counter() - t0

print(f"Train: {X_train_enc.shape[0]:,} rows, {X_train_enc.shape[1]} features")
print(f"Test:  {X_test_enc.shape[0]:,} rows,  {X_test_enc.shape[1]} features")
print(f"Train positive rate: {y_train.mean():.3f}")
print(f"Test  positive rate: {y_test.mean():.3f}")
print(f"Any nulls in train:  {X_train_enc.isnull().sum().sum()}")
print(f"Any nulls in test:   {X_test_enc.isnull().sum().sum()}")
print(f"\n[TIMING] Feature Engineering (train): {t_fe_train_cpu:.3f}s")
print(f"[TIMING] Feature Engineering (test):  {t_fe_test_cpu:.3f}s")


## 4. Model 1: Logistic Regression

In this section I define a Logistic Regression pipeline, tune its hyperparameters via cross-validation on the full training set, and evaluate performance on the held-out test set via a threshold sweep optimizing for F1 score.

Logistic Regression models the log-odds of escalation as a linear combination of input features, applying a sigmoid transformation to produce a probability estimate between 0 and 1. It is chosen as the first model for two reasons. First, it serves as an interpretable baseline. Its coefficients directly quantify the direction and magnitude of each feature's contribution to escalation risk, which is critical in regulated financial services environments where predictions may need to be explained to auditors or regulators. Second, it provides well-calibrated probability estimates, making threshold adjustment transparent and defensible.

In a complaint management context, a model that can be interrogated is often more operationally valuable than a marginally more accurate black box. Logistic Regression allows a compliance officer to understand the model and its decisions and to derive more actionable insights. For example, the association between explanation-only responses and a higher log-odds of escalation than monetary relief responses, a finding that directly informs response strategy. Logistic regressions are also computationally efficient, enabling real-time scoring of incoming complaints at scale.

The 4:1 class imbalance is addressed via the `class_weight='balanced'` parameter, which inversely weights each class by its frequency during training. This penalizes misclassification of the minority class more heavily, preventing the model from defaulting to predicting the majority class. This approach is preferred over synthetic oversampling as it achieves a comparable effect without altering the training distribution.

Hyperparameters are tuned via `GridSearchCV` with 5-fold stratified cross-validation on the full training set, optimizing for F1 score. The key hyperparameters are the regularization strength `C`, which controls the trade-off between fitting the training data and keeping coefficients small, and the penalty type (`l1` or `l2`), where L1 regularization induces sparsity by driving irrelevant coefficients to zero and L2 penalizes large coefficients without eliminating features entirely.

In [ ]:
# logistic regression pipeline: impute residual nulls + scale + classify
lr_pipeline = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('scaler',     StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

In [ ]:
# hyperparameter tuning via GridSearch with stratified 5-fold CV
param_grid_lr = {
    'classifier__C':            [0.001, 0.01, 0.1, 1, 10],
    'classifier__penalty':      ['l1', 'l2'],
    'classifier__class_weight': ['balanced'],
    'classifier__solver':       ['liblinear']
}

cv_lr = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search_lr = GridSearchCV(
    lr_pipeline, param_grid_lr,
    cv=cv_lr, scoring='f1',
    n_jobs=1, verbose=1
)

t0 = time.perf_counter()
search_lr.fit(X_train_enc, y_train)
t_lr_cv_cpu = time.perf_counter() - t0

# best estimator is already retrained on full training set via refit=True (default)
t0 = time.perf_counter()
y_prob_lr = search_lr.best_estimator_.predict_proba(X_test_enc)[:, 1]
t_lr_infer_cpu = time.perf_counter() - t0

best_f1_lr, best_t_lr = 0, 0
for t in np.arange(0.10, 0.70, 0.005):
    f1 = f1_score(y_test, (y_prob_lr >= t).astype(int))
    if f1 > best_f1_lr:
        best_f1_lr = f1
        best_t_lr  = t

y_pred_lr = (y_prob_lr >= best_t_lr).astype(int)

print(f"Best params:  {search_lr.best_params_}")
print(f"CV F1:        {search_lr.best_score_:.4f}")
print(f"Test F1:      {best_f1_lr:.4f}  at threshold {best_t_lr:.3f}")
print(f"Precision:    {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall:       {recall_score(y_test, y_pred_lr):.4f}")
print(f"ROC-AUC:      {roc_auc_score(y_test, y_prob_lr):.4f}")
print(f"\n[TIMING] LR GridSearchCV (50 fits, 5-fold): {t_lr_cv_cpu:.2f}s  ({t_lr_cv_cpu/60:.1f} min)")
print(f"[TIMING] LR Inference (test set):           {t_lr_infer_cpu:.4f}s")


**Findings & Comments:**

The Logistic Regression GridSearchCV (10 parameter combinations × 5-fold stratified CV = 50 fits) completed in `[TIMING] LR GridSearchCV` seconds as printed above. The best hyperparameters, cross-validation F1, test F1, and ROC-AUC are reported by the cell output. Inference on the 64,286-record test set completes in milliseconds. These timing values are the CPU baseline for the GPU speedup comparison in `GPU_Solution.ipynb`.


## 5. Model 2: XGBoost

In this section I define an XGBoost pipeline, tune its hyperparameters via randomized search with 5-fold stratified cross-validation on the full training set, and evaluate performance on the held-out test set via a threshold sweep optimizing for F1 score.

XGBoost is a gradient boosted tree ensemble that builds an additive sequence of decision trees, where each tree corrects the residual errors of the previous one. Unlike Logistic Regression, it captures non-linear relationships and feature interactions without requiring explicit engineering, making it well suited to a dataset where escalation risk is driven by complex combinations of response type, company behavior, consumer demographics, and complaint characteristics. Regularization parameters `reg_alpha` (L1) and `reg_lambda` (L2) control model complexity at the tree level, and `scale_pos_weight` directly addresses class imbalance by upweighting the minority class during training.

While Logistic Regression provides interpretability, XGBoost is chosen as the primary model because complaint escalation is unlikely to be a linear function of the available features. A consumer who submitted a long narrative, received an explanation-only response from a high-dispute-rate company, and is tagged as an Older American represents a compounding risk profile that a linear model cannot fully capture. XGBoost's ability to model these interactions makes it more appropriate for operational deployment where predictive accuracy directly translates to resource allocation and intervention decisions.

The 4:1 class imbalance is addressed via the `scale_pos_weight` hyperparameter, set to 4 to reflect the inverse class ratio. This instructs the model to treat each positive (disputed) instance as four times more important than a negative instance during training, improving sensitivity to the minority class without altering the training data distribution.

Hyperparameters are tuned via `RandomizedSearchCV` with 5-fold stratified cross-validation optimizing for F1 score across 60 iterations. The key parameters and their roles are as follows: `max_depth` controls tree depth and thus model complexity; `learning_rate` determines the contribution of each tree to the ensemble (lower values require more trees but generalize better); `n_estimators` sets the number of trees; `min_child_weight` controls the minimum number of samples required in a leaf node, acting as a regularizer against overfitting on sparse feature combinations; `subsample` and `colsample_bytree` introduce stochasticity by training each tree on a random row and column subsample respectively, reducing variance; and `reg_alpha` and `reg_lambda` provide L1 and L2 regularization on leaf weights.

**The hyperparameter grid was refined iteratively across multiple tuning rounds, progressively narrowing the search space around the best-performing configurations. The final grid shown below reflects the refined ranges identified through this process. The best estimator identified by `RandomizedSearchCV` is automatically retrained on the full training set via `refit=True` and serves as the final model for evaluation and deployment.**

In [ ]:
# XGBoost pipeline: impute residual nulls only (scale-invariant model)
xgb_pipeline = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('classifier', XGBClassifier(
        random_state=42, eval_metric='logloss', use_label_encoder=False
    ))
])

In [ ]:
# hyperparameter tuning via RandomizedSearch with stratified 5-fold CV
param_grid_xgb = {
    'classifier__n_estimators':     [400, 500, 600, 800],
    'classifier__max_depth':        [4, 5, 6],
    'classifier__learning_rate':    [0.02, 0.03, 0.05],
    'classifier__scale_pos_weight': [3.5, 4, 4.5],
    'classifier__min_child_weight': [5, 10, 15],
    'classifier__subsample':        [0.55, 0.6, 0.65, 0.7],
    'classifier__colsample_bytree': [0.6, 0.65, 0.7],
    'classifier__reg_alpha':        [0, 0.1, 1],
    'classifier__reg_lambda':       [3, 5, 7]
}

cv_xgb = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search_xgb = RandomizedSearchCV(
    xgb_pipeline, param_grid_xgb,
    n_iter=60, cv=cv_xgb, scoring='f1',
    random_state=42, n_jobs=1, verbose=1
)

t0 = time.perf_counter()
search_xgb.fit(X_train_enc, y_train)
t_xgb_cv_cpu = time.perf_counter() - t0

t0 = time.perf_counter()
y_prob_xgb = search_xgb.best_estimator_.predict_proba(X_test_enc)[:, 1]
t_xgb_infer_cpu = time.perf_counter() - t0

best_f1_xgb, best_t_xgb = 0, 0
for t in np.arange(0.10, 0.70, 0.005):
    f1 = f1_score(y_test, (y_prob_xgb >= t).astype(int))
    if f1 > best_f1_xgb:
        best_f1_xgb = f1
        best_t_xgb  = t

y_pred_xgb = (y_prob_xgb >= best_t_xgb).astype(int)

print(f"Best params:  {search_xgb.best_params_}")
print(f"CV F1:        {search_xgb.best_score_:.4f}")
print(f"Test F1:      {best_f1_xgb:.4f}  at threshold {best_t_xgb:.3f}")
print(f"Precision:    {precision_score(y_test, y_pred_xgb):.4f}")
print(f"Recall:       {recall_score(y_test, y_pred_xgb):.4f}")
print(f"ROC-AUC:      {roc_auc_score(y_test, y_prob_xgb):.4f}")
print(f"\n[TIMING] XGB RandomizedSearchCV (300 fits, 5-fold): {t_xgb_cv_cpu:.2f}s  ({t_xgb_cv_cpu/60:.1f} min)")
print(f"[TIMING] XGB Inference (test set):                  {t_xgb_infer_cpu:.4f}s")


**Findings & Comments:**

The XGBoost RandomizedSearchCV (60 iterations × 5-fold stratified CV = 300 fits) completed in `[TIMING] XGB RandomizedSearchCV` seconds as printed above. This is the most compute-intensive stage of the pipeline and the primary target for GPU acceleration. The best hyperparameters, cross-validation F1, test F1, and ROC-AUC are reported by the cell output. Inference on the test set completes in milliseconds. These timing values are the CPU baseline for the GPU speedup comparison in `GPU_Solution.ipynb`.


# 8. Performance Benchmarking (CPU Baseline)

This section measures wall-clock time for the compute-heavy stages of the pipeline across five dataset-size fractions. The same benchmark is run in `GPU_Solution.ipynb` to compute speedup ratios.

**Methodology:**
- Five training-set fractions: 10%, 25%, 50%, 75%, 100%
- At each fraction: time feature engineering, a single LR fit, a single XGB fit, and inference on the full test set
- Single fits use the best hyperparameters found by the CV searches above
- `tree_method='hist'` is set explicitly on both CPU and GPU notebooks so the tree algorithm is identical and results are comparable
- No cross-validation in the sweep (CV timing is captured separately in the cells above)


In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# Extract best hyperparams from CV results
best_C       = search_lr.best_params_['classifier__C']
best_penalty = search_lr.best_params_['classifier__penalty']

best_xgb_raw = {k.replace('classifier__', ''): v
                for k, v in search_xgb.best_params_.items()}

fractions = [0.10, 0.25, 0.50, 0.75, 1.00]
results_cpu = []

for frac in fractions:
    n = int(len(X_train_raw) * frac)
    X_sub = X_train_raw.iloc[:n].copy()
    y_sub = y_train.iloc[:n].copy()
    row   = {'fraction': frac, 'n_train': n}

    # --- Feature Engineering ---
    t0 = time.perf_counter()
    X_sub_enc, enc_sub = feature_engineering(X_sub, y=y_sub)
    row['cpu_fe_s'] = time.perf_counter() - t0

    X_test_enc_b, _ = feature_engineering(X_test_raw, fit_encoders=enc_sub)

    # --- LR single fit (best params, no CV) ---
    lr_b = Pipeline([
        ('imputer',    SimpleImputer(strategy='median')),
        ('scaler',     StandardScaler()),
        ('classifier', LogisticRegression(
            C=best_C, penalty=best_penalty,
            class_weight='balanced', solver='liblinear',
            random_state=42, max_iter=1000
        ))
    ])
    t0 = time.perf_counter()
    lr_b.fit(X_sub_enc, y_sub)
    row['cpu_lr_train_s'] = time.perf_counter() - t0

    t0 = time.perf_counter()
    lr_b.predict_proba(X_test_enc_b)
    row['cpu_lr_infer_s'] = time.perf_counter() - t0

    # --- XGB single fit (best params, hist on CPU, no CV) ---
    xgb_b = Pipeline([
        ('imputer',    SimpleImputer(strategy='median')),
        ('classifier', XGBClassifier(
            **best_xgb_raw,
            random_state=42, eval_metric='logloss',
            tree_method='hist', device='cpu'
        ))
    ])
    t0 = time.perf_counter()
    xgb_b.fit(X_sub_enc, y_sub)
    row['cpu_xgb_train_s'] = time.perf_counter() - t0

    t0 = time.perf_counter()
    xgb_b.predict_proba(X_test_enc_b)
    row['cpu_xgb_infer_s'] = time.perf_counter() - t0

    results_cpu.append(row)
    print(f"frac={frac:.0%}  n={n:,}  |  FE={row['cpu_fe_s']:.2f}s  "
          f"LR_train={row['cpu_lr_train_s']:.2f}s  "
          f"XGB_train={row['cpu_xgb_train_s']:.2f}s  "
          f"LR_infer={row['cpu_lr_infer_s']:.4f}s  "
          f"XGB_infer={row['cpu_xgb_infer_s']:.4f}s")

df_cpu = pd.DataFrame(results_cpu)
print("\n=== CPU Baseline Timing Results ===")
print(df_cpu.to_string(index=False))


## 8.1 CPU Timing Summary

The table above records wall-clock seconds for each operation at each dataset size. These numbers feed directly into the speedup tables in `GPU_Solution.ipynb`.

**Full CV timings (100% data):**


In [ ]:
print(f"LR  GridSearchCV  (50 fits):  {t_lr_cv_cpu:.1f}s  ({t_lr_cv_cpu/60:.1f} min)")
print(f"XGB RandomizedCV (300 fits): {t_xgb_cv_cpu:.1f}s  ({t_xgb_cv_cpu/60:.1f} min)")
print(f"LR  Inference (test set):    {t_lr_infer_cpu*1000:.1f} ms")
print(f"XGB Inference (test set):    {t_xgb_infer_cpu*1000:.1f} ms")
